# Test the functionality for crawling stocks

In [1]:
import stock_lib as sl
from datetime import date

## Crawl

In [31]:
df_stock = sl.crawl_stock("NVDA", date(2026,8,1), date(2026,8,24))
df_stock.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-08-03 00:00:00-04:00,197.690002,208.740005,196.850006,206.639999,128406900,0.0,0.0
2026-08-04 00:00:00-04:00,211.300003,213.059998,209.050003,211.940002,134922000,0.0,0.0
2026-08-05 00:00:00-04:00,216.860001,222.220001,216.399994,219.220001,158187400,0.0,0.0
2026-08-06 00:00:00-04:00,221.529999,223.630005,217.000000,218.990005,113940600,0.0,0.0
2026-08-07 00:00:00-04:00,221.539993,224.759995,220.660004,223.960007,105669400,0.0,0.0


In [32]:
df_stock

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-08-03 00:00:00-04:00,197.690002,208.740005,196.850006,206.639999,128406900,0.0,0.0
2026-08-04 00:00:00-04:00,211.300003,213.059998,209.050003,211.940002,134922000,0.0,0.0
2026-08-05 00:00:00-04:00,216.860001,222.220001,216.399994,219.220001,158187400,0.0,0.0
2026-08-06 00:00:00-04:00,221.529999,223.630005,217.000000,218.990005,113940600,0.0,0.0
2026-08-07 00:00:00-04:00,221.539993,224.759995,220.660004,223.960007,105669400,0.0,0.0
2026-08-10 00:00:00-04:00,223.399994,224.139999,216.770004,217.550003,115846600,0.0,0.0
2026-08-11 00:00:00-04:00,222.169998,222.199997,216.199997,217.500000,101273100,0.0,0.0
2026-08-12 00:00:00-04:00,221.039993,225.100006,220.199997,224.089996,108783600,0.0,0.0
2026-08-13 00:00:00-04:00,225.059998,227.229996,223.710007,225.300003,98867200,0.0,0.0


In [5]:
comp = sl.crawl_company("MSFT")

In [6]:
comp

{'address1': 'One Microsoft Way',
 'city': 'Redmond',
 'state': 'WA',
 'zip': '98052-6399',
 'country': 'United States',
 'phone': '425 882 8080',
 'website': 'https://www.microsoft.com',
 'industry': 'Software - Infrastructure',
 'industryKey': 'software-infrastructure',
 'industryDisp': 'Software - Infrastructure',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': "Microsoft Corporation, a technology company, develops and supports a portfolio of technology solutions for individuals and businesses worldwide. Its products include operating systems, server applications, business solution applications, software development tools, desktop and server management tools, and video games; and devices, such as PCs, tablets, gaming and entertainment consoles, other intelligent devices, and related accessories. The company's Productivity and Business Processes segment offers Microsoft 365 Commercial, Enterprise Mobility + Security, Power BI,

## Insert into database

In [34]:
sl.upsert_company(comp)

NameError: name 'comp' is not defined

In [35]:
sl.upsert_stock("NVDA", df_stock)

{'status': 'Success', 'rows': 15}

## Select functions

In [9]:
sl.get_company("MSFT")

[{'symbol': 'MSFT',
  'name': 'Microsoft',
  'sector': 'Technology',
  'industry': 'Software - Infrastructure',
  'description': "Microsoft Corporation, a technology company, develops and supports a portfolio of technology solutions for individuals and businesses worldwide. Its products include operating systems, server applications, business solution applications, software development tools, desktop and server management tools, and video games; and devices, such as PCs, tablets, gaming and entertainment consoles, other intelligent devices, and related accessories. The company's Productivity and Business Processes segment offers Microsoft 365 Commercial, Enterprise Mobility + Security, Power BI, Exchange, SharePoint, Microsoft Teams, Microsoft 365 Security and Compliance, Microsoft 365 Copilot, and Windows Commercial on-premises and Office licensed on-premises. This segment also provides Microsoft 365 Consumer products and cloud services; LinkedIn, including talent solutions, marketing

In [36]:
df_stock_db = sl.get_stock("NVDA", date(2026, 8, 17), date(2026, 8, 24))
df_stock_db

,symbol,date,open,close,high,low,volume,dividends,stock_split,percent_change_close
0,NVDA,2026-08-17,225.979996,225.009995,227.919998,224.860001,93678700,0.0,0.0,NaN
1,NVDA,2026-08-18,220.449997,219.740005,221.639999,218.690002,103128200,0.0,0.0,-0.023421
2,NVDA,2026-08-19,221.669998,217.559998,222.869995,216.759995,96797300,0.0,0.0,-0.009921
3,NVDA,2026-08-20,218.360001,216.850006,219.860001,215.660004,92457000,0.0,0.0,-0.003263
4,NVDA,2026-08-21,218.419998,214.720001,218.740005,214.500000,98545600,0.0,0.0,-0.009822


In [ ]:
df_desc = df_stock_db.describe()
sig = sl.adaptive_sig(date(2026, 8, 1), date(2026, 8, 10))
column = "close"
std_dev = df_desc.loc["std", "percent_change_"+column]
center = df_desc.loc["mean", "percent_change_"+column]
lower_range = center - (sig*std_dev)
upper_range = center + (sig*std_dev)

In [28]:
print(f"low: {lower_range}")
print(f"upper: {upper_range}")

low: -0.018784517289865826
upper: 0.03999354775121051


In [27]:
df_stock_db[
    (df_stock_db["percent_change_"+column] < lower_range) | 
    (df_stock_db["percent_change_"+column] > upper_range)
]

,symbol,date,open,close,high,low,volume,dividends,stock_split,percent_change_close,percent_increase_close
5,NVDA,2026-08-10,223.399994,217.550003,224.139999,216.770004,115846600,0.0,0.0,-0.028621,False


## Statistics

In [37]:
z= sl.adaptive_sig(date(2026, 8, 17), date(2026, 8, 24))
df_sig = sl.spot_significant_dates(df_stock_db, "close", mean=True, sig=z, limit=10, increase=None)
df_sig

,symbol,date,open,close,high,low,volume,dividends,stock_split,percent_change_close,percent_increase_close
1,NVDA,2026-08-18,220.449997,219.740005,221.639999,218.690002,103128200,0.0,0.0,0.023421,False


In [10]:
df_sig_1 = sl.spot_significant_dates(df_stock_db, "close", mean=True, sig=1, limit=10, increase = None)
df_sig_1

,symbol,date,open,close,high,low,volume,dividends,stock_split,percent_change,percent_increase
4,NVDA,2026-06-05,214.529999,205.100006,214.869995,204.330002,219655500,0.0,0.0,0.062014,False
15,NVDA,2026-06-23,202.169998,200.039993,203.770004,200.000000,153496200,0.0,0.0,0.041265,False
7,NVDA,2026-06-10,204.429993,200.419998,207.220001,199.919998,161746600,0.0,0.0,0.037322,False
2,NVDA,2026-06-03,221.461887,214.500000,222.560613,214.260274,160907000,0.0,0.0,0.036218,False
10,NVDA,2026-06-15,208.919998,212.449997,212.710007,208.339996,149936700,0.0,0.0,0.035382,True
13,NVDA,2026-06-18,207.330002,210.690002,211.389999,206.500000,241272000,0.0,0.0,0.029514,True
20,NVDA,2026-06-30,197.240005,200.089996,200.630005,195.110001,166476700,0.0,0.0,0.026260,True
8,NVDA,2026-06-11,201.490005,204.869995,205.660004,199.539993,158643200,0.0,0.0,0.022203,True
